# NZ Meal Cost Optimizer

Find the cheapest Pak'nSave for your recipe.

1. Enter your address
2. Enter a dish name
3. Compare prices across nearby stores

In [ ]:
import cloudscraper
import pandas as pd
import requests
import math
from pathlib import Path

stores = pd.read_csv("../data/paknsave_stores.csv")
print(f"Loaded {len(stores)} Pak'nSave stores")

Loaded 60 Pak'nSave stores


In [2]:
BASE = "https://api-prod.prod.fsniwaikato.kiwi/prod"

class PaknSaveAPI:
    def __init__(self):
        self.scraper = cloudscraper.create_scraper()
        self._token = None

    def _ensure_token(self):
        if self._token:
            return
        r = self.scraper.post(
            f"{BASE}/mobile/user/login/guest",
            json={"banner": "PNS"},
            headers={"User-Agent": "PAKnSAVEApp/4.32.0", "Content-Type": "application/json"},
        )
        r.raise_for_status()
        self._token = r.json()["access_token"]
        self._auth = {
            "Authorization": f"Bearer {self._token}",
            "access_token": self._token,
            "User-Agent": "PAKnSAVEApp/4.32.0",
            "Content-Type": "application/json",
        }

    def search_products(self, store_id: str, query: str):
        self._ensure_token()
        r = self.scraper.post(
            f"{BASE}/mobile/ecomm-products/PNS/{store_id}/search?q={query}",
            headers=self._auth, json=[],
        )
        if r.status_code == 200:
            return r.json()
        return None

api = PaknSaveAPI()
print("API client ready")

API client ready


In [3]:
def geocode(address):
    r = requests.get(
        "https://nominatim.openstreetmap.org/search",
        headers={"User-Agent": "NZMealCostOptimizer/1.0"},
        params={"q": address, "format": "json", "limit": 1},
    )
    if r.status_code == 200 and r.json():
        loc = r.json()[0]
        return float(loc["lat"]), float(loc["lon"])
    return None, None

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2) ** 2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2) ** 2
    return R * 2 * math.asin(math.sqrt(a))

def find_nearby(user_lat, user_lon, radius_km=5):
    df = stores.copy()
    df["distance_km"] = df.apply(
        lambda r: haversine(user_lat, user_lon, r["latitude"], r["longitude"]), axis=1,
    )
    return df[df["distance_km"] <= radius_km].sort_values("distance_km")

In [4]:
DISH_INGREDIENTS = {
    "spaghetti bolognese": ["beef mince", "spaghetti pasta", "canned tomatoes", "onion", "carrot", "garlic", "mixed herbs"],
    "chicken stir fry": ["chicken breast", "stir fry vegetables", "soy sauce", "rice noodles"],
    "beef stir fry": ["beef strips", "stir fry vegetables", "soy sauce", "rice noodles"],
    "roast lamb": ["lamb roast", "potato", "carrot", "broccoli", "stock"],
    "chicken curry": ["chicken thigh", "curry paste", "coconut milk", "rice", "onion"],
    "beef curry": ["diced beef", "curry paste", "coconut milk", "rice", "onion"],
    "fish and chips": ["fish fillet", "potato", "oil"],
    "nachos": ["beef mince", "tortilla chips", "cheese", "beans", "sour cream"],
    "pumpkin soup": ["pumpkin", "onion", "cream", "stock", "bread"],
    "tacos": ["beef mince", "taco shells", "lettuce", "tomato", "cheese", "sour cream"],
    "lamb chops": ["lamb chops", "potato", "mint sauce"],
    "butter chicken": ["chicken thigh", "butter chicken sauce", "rice", "cream"],
    "lasagne": ["beef mince", "lasagne sheets", "cheese", "canned tomatoes", "milk", "butter", "flour"],
    "shepherd's pie": ["beef mince", "potato", "carrot", "peas", "stock"],
    "pizza": ["pizza base", "pizza sauce", "cheese", "pepperoni"],
    "vegie stir fry": ["stir fry vegetables", "tofu", "soy sauce", "rice noodles", "garlic"],
    "frittata": ["eggs", "potato", "onion", "cheese", "milk"],
    "pancakes": ["flour", "eggs", "milk", "sugar", "butter"],
    "chicken soup": ["chicken breast", "carrot", "onion", "celery", "stock", "pasta"],
    "tomato pasta": ["pasta", "canned tomatoes", "garlic", "olive oil", "mixed herbs", "cheese"],
    "chicken katsu": ["chicken breast", "flour", "eggs", "bread", "rice", "katsu sauce"],
}

def get_ingredients(dish_name):
    return DISH_INGREDIENTS.get(dish_name.lower().strip(), [dish_name])

def search_ingredient(store_id, ingredient):
    results = api.search_products(store_id, ingredient)
    if not results:
        return None
    products = results.get("products", [])
    if not products:
        return None
    p = products[0]
    price_cents = p.get("price")
    if price_cents is None or price_cents <= 0:
        return None
    return {
        "name": p["name"],
        "brand": p.get("brand", ""),
        "price": price_cents / 100,
        "units": p.get("units", ""),
    }

## Run the optimizer

Edit the two variables below and run this cell.

In [5]:
# ═══════ YOUR INPUTS ═══════
USER_ADDRESS = "Botany Town Centre, Auckland 2013"
DISH_NAME = "spaghetti bolognese"
MAX_DISTANCE_KM = 5
# ═══════════════════════════

user_lat, user_lon = geocode(USER_ADDRESS)
if user_lat is None:
    raise SystemExit(f"Could not geocode: {USER_ADDRESS}")
print(f"Address: {USER_ADDRESS}")
print(f"Location: {user_lat:.5f}, {user_lon:.5f}\n")

nearby = find_nearby(user_lat, user_lon, MAX_DISTANCE_KM)
if nearby.empty:
    raise SystemExit(f"No stores within {MAX_DISTANCE_KM}km")

print(f"Nearby stores ({len(nearby)} within {MAX_DISTANCE_KM}km):")
for _, s in nearby.iterrows():
    print(f"  {s['name']:30s} {s['distance_km']:.2f} km")
print()

ingredients = get_ingredients(DISH_NAME)
print(f"Dish: {DISH_NAME}")
print(f"Ingredients: {', '.join(ingredients)}\n")

all_results = []
for _, store in nearby.iterrows():
    store_id = store["store_id"]
    store_name = store["name"]
    print(f"--- {store_name} ---")
    total = 0.0
    for ing in ingredients:
        result = search_ingredient(store_id, ing)
        if result:
            all_results.append({**result, "store": store_name, "ingredient": ing, "distance_km": store["distance_km"]})
            print(f"  {ing:25s} ${result['price']:>5.2f}  {result['name']} ({result['units']})")
            total += result["price"]
        else:
            print(f"  {ing:25s}  NOT FOUND")
    print(f"  {'TOTAL':25s} ${total:.2f}\n")

if all_results:
    df = pd.DataFrame(all_results)
    summary = df.groupby("store").agg(
        total_cost=("price", "sum"),
        items_found=("ingredient", "count"),
        distance_km=("distance_km", "first"),
    ).sort_values("total_cost")
    print("=" * 65)
    print("COST COMPARISON")
    print("=" * 65)
    print(summary.to_string())
    print()
    best = summary.index[0]
    best_total = summary["total_cost"].iloc[0]
    print(f"Cheapest: {best} - ${best_total:.2f} total")

Address: Botany Town Centre, Auckland 2013
Location: -36.93226, 174.91315

Nearby stores (3 within 5km):
  Botany                         0.32 km
  Ormiston                       3.61 km
  Highland Park                  3.65 km

Dish: spaghetti bolognese
Ingredients: beef mince, spaghetti pasta, canned tomatoes, onion, carrot, garlic, mixed herbs

--- Botany ---
  beef mince                $18.99  NZ Beef Mince (kg)
  spaghetti pasta           $ 1.19  Spaghetti (400g)
  canned tomatoes           $ 0.89  Chopped Tomatoes in Juice (400g)
  onion                     $ 1.69  Brown Onions (kg)
  carrot                    $ 1.99  Carrots (kg)
  garlic                    $ 2.29  Crushed Garlic (230g)
  mixed herbs               $ 1.99  Naturals Eco Mixed Herbs (10g)
  TOTAL                     $29.03

--- Ormiston ---
  beef mince                $14.99  NZ Beef Mince (kg)
  spaghetti pasta           $ 1.29  Spaghetti (400g)
  canned tomatoes           $ 0.89  Chopped Tomatoes in Juice (400g)


In [6]:
# Show the cheapest store's itemized list
if all_results:
    df = pd.DataFrame(all_results)
    cheapest_store = df.groupby("store")["price"].sum().idxmin()
    items = df[df["store"] == cheapest_store][["ingredient", "name", "units", "price"]]
    display(items.style.format({"price": "${:.2f}"}).hide(axis="index"))

ingredient,name,units,price
beef mince,NZ Beef Mince,kg,$18.99
spaghetti pasta,Spaghetti,400g,$1.19
canned tomatoes,Chopped Tomatoes in Juice,400g,$0.89
onion,Brown Onions,kg,$1.69
carrot,Carrots,kg,$1.99
garlic,Crushed Garlic,230g,$2.29
mixed herbs,Naturals Eco Mixed Herbs,10g,$1.99
